In [5]:
import astropy
import astroquery.simbad
import numpy as np
from astropy.table import Table
from astropy.io import ascii
import astropy.units as u
from astropy.coordinates import SkyCoord
import pyarrow
import pyvo
from collections import Counter
from crossmatching import NEACatalog, SimbadIdSupplier
import timeit
import matplotlib.pyplot as plt
import pandas
import os
import numpy.ma as ma

%load_ext autoreload
%autoreload 1
%aimport crossmatching
from crossmatching import Crossmatcher


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
cm = Crossmatcher(NEACatalog(), SimbadIdSupplier())
cm.load_catalog(from_file="pscomppars.txt")
input_t = ascii.read("./input/HPIC_LC4_combined_d50.txt")
name_list = input_t["star_name"]  

In [ ]:
_ = cm.load_alternate_ids(name_list, from_file="alternate_ids.txt")

In [ ]:
cm.find_duplicates(input_t)

input_ids,duplicate_names,appearances
str28,object,int64
* 70 Oph A,"['* 70 Oph A', 'TIC 1674663309']",2
GAIA DR3 3111066471465385216,"['GAIA DR3 3111066471465385216', 'TIC 262722576']",2
TIC 143203922,"['TIC 143203922', 'TIC 471012437']",2
TIC 155315739,"['TIC 155315739', 'TIC 471015557']",2
TIC 436933290,"['TIC 436933290', 'TIC 436933294']",2
TIC 437127546,"['TIC 437127546', 'TIC 471012765']",2
TIC 471012273,"['TIC 471012273', 'TIC 98755656']",2


In [ ]:
input_t[np.isin(input_t["star_name"],['GAIA DR3 3111066471465385216', 'TIC 262722576']	)]

star_name,sy_dist,st_spectype,st_rad,st_teff,st_mass,st_age,ra,dec,sy_vmag,sy_jmag,sy_hmag,sy_kmag,known_binary_fl,gaia_binary_fl,WDSsep,wds_deltamag
str29,float64,str18,float64,str18,float64,str18,float64,float64,str18,str18,str18,str18,str5,str4,str18,str19
TIC 262722576,46.065765380859375,G8V,0.8475582853524161,5420.0,0.8998814225196838,4.972176551818848,138.44585937883,4.4208484,8.67733383178711,8.67733383178711,8.67733383178711,8.67733383178711,0,0,null,null
GAIA DR3 3111066471465385216,47.201908111572266,K0/1V,0.8993623863594754,5199.0,0.760515,13.498753,109.421549,0.1347244,8.800736427307129,8.800736427307129,8.800736427307129,8.800736427307129,0,0,null,null


In [ ]:
nodupes = cm.remove_duplicates(input_t)
print(f"{len(input_t) = }\n{len(nodupes) = }\n{len(input_t)-len(nodupes) = }")

Removed Rows with indecies and names: 599, 7161, 3836, 1368, 6618, 5097, 12288
len(input_t) = 14571
len(nodupes) = 14564
len(input_t)-len(nodupes) = 7


In [9]:
result = cm.combined_crossmatch(input_t, input_starname_key="star_name")

In [34]:
mass_related_cols = [i for i in result.colnames if "mass" in i or "msini" in i]

prox_cen = (result["pl_name"] == "Proxima Cen b")
result["pl_name", *mass_related_cols][prox_cen]

from collections import Counter
# Counter(cm.catalog_table["pl_trueobliq"])
(cm.catalog_table["st_spectype"].mask).sum()
cm.catalog_table.sort("pl_rade", reverse=True)
cm.catalog_table[cm.catalog_table["st_spectype"].mask]

objectid,pl_name,pl_letter,hostid,hostname,hd_name,hip_name,tic_id,disc_pubdate,disc_year,disc_method,discoverymethod,disc_locale,disc_facility,disc_instrument,disc_telescope,disc_refname,ra,raerr1,raerr2,rasymerr,rastr,ra_solnid,ra_reflink,dec,decerr1,decerr2,decsymerr,decstr,dec_solnid,dec_reflink,glon,glonerr1,glonerr2,glonsymerr,glonstr,glon_solnid,glon_reflink,glat,glaterr1,glaterr2,glatsymerr,glatstr,glat_solnid,glat_reflink,elon,elonerr1,elonerr2,elonsymerr,elonstr,elon_solnid,elon_reflink,elat,elaterr1,elaterr2,elatsymerr,elat_solnid,elat_reflink,elatstr,pl_orbper,pl_orbpererr1,pl_orbpererr2,pl_orbpersymerr,pl_orbperlim,pl_orbperstr,pl_orbperformat,pl_orbper_solnid,pl_orbper_reflink,pl_orblpererr1,pl_orblper,pl_orblpererr2,pl_orblpersymerr,pl_orblperlim,pl_orblperstr,pl_orblperformat,pl_orblper_solnid,pl_orblper_reflink,pl_orbsmax,pl_orbsmaxerr1,pl_orbsmaxerr2,pl_orbsmaxsymerr,pl_orbsmaxlim,pl_orbsmaxstr,pl_orbsmaxformat,pl_orbsmax_solnid,pl_orbsmax_reflink,pl_orbincl,pl_orbinclerr1,pl_orbinclerr2,pl_orbinclsymerr,pl_orbincllim,pl_orbinclstr,pl_orbinclformat,pl_orbincl_solnid,pl_orbincl_reflink,pl_orbtper,pl_orbtpererr1,pl_orbtpererr2,pl_orbtpersymerr,pl_orbtperlim,pl_orbtperstr,pl_orbtperformat,pl_orbtper_solnid,pl_orbtper_reflink,pl_orbeccen,pl_orbeccenerr1,pl_orbeccenerr2,pl_orbeccensymerr,pl_orbeccenlim,pl_orbeccenstr,pl_orbeccenformat,pl_orbeccen_solnid,pl_orbeccen_reflink,pl_eqt,pl_eqterr1,pl_eqterr2,pl_eqtsymerr,pl_eqtlim,pl_eqtstr,pl_eqtformat,pl_eqt_solnid,pl_eqt_reflink,pl_occdep,pl_occdeperr1,pl_occdeperr2,pl_occdepsymerr,pl_occdeplim,pl_occdepstr,pl_occdepformat,pl_occdep_solnid,pl_occdep_reflink,pl_insol,pl_insolerr1,pl_insolerr2,pl_insolsymerr,pl_insollim,pl_insolstr,pl_insolformat,pl_insol_solnid,pl_insol_reflink,pl_dens,pl_denserr1,sy_umagerr1,sy_umagerr2,sy_umaglim,sy_umagsymerr,sy_umagstr,sy_umagformat,sy_umag_solnid,sy_umag_reflink,sy_rmag,sy_rmagerr1,sy_rmagerr2,sy_rmaglim,sy_rmagsymerr,sy_rmagstr,sy_rmagformat,sy_rmag_solnid,sy_rmag_reflink,sy_imag,sy_imagerr1,sy_imagerr2,sy_imaglim,sy_imagsymerr,sy_imagstr,sy_imagformat,sy_imag_solnid,sy_imag_reflink,sy_zmag,sy_zmagerr1,sy_zmagerr2,sy_zmaglim,sy_zmagsymerr,sy_zmagstr,sy_zmagformat,sy_zmag_solnid,sy_zmag_reflink,sy_w1mag,sy_w1magerr1,sy_w1magerr2,sy_w1maglim,sy_w1magsymerr,sy_w1magstr,sy_w1magformat,sy_w1mag_solnid,sy_w1mag_reflink,sy_w2mag,sy_w2magerr1,sy_w2magerr2,sy_w2maglim,sy_w2magsymerr,sy_w2magstr,sy_w2magformat,sy_w2mag_solnid,sy_w2mag_reflink,sy_w3mag,sy_w3magerr1,sy_w3magerr2,sy_w3maglim,sy_w3magsymerr,sy_w3magstr,sy_w3magformat,sy_w3mag_solnid,sy_w3mag_reflink,sy_w4mag,sy_w4magerr1,sy_w4magerr2,sy_w4maglim,sy_w4magsymerr,sy_w4magstr,sy_w4magformat,sy_w4mag_solnid,sy_w4mag_reflink,sy_gmag,sy_gmagerr1,sy_gmagerr2,sy_gmaglim,sy_gmagsymerr,sy_gmagstr,sy_gmagformat,sy_gmag_solnid,sy_gmag_reflink,sy_gaiamag,sy_gaiamagerr1,sy_gaiamagerr2,sy_gaiamaglim,sy_gaiamagsymerr,sy_gaiamagstr,sy_gaiamagformat,sy_gaiamag_solnid,sy_gaiamag_reflink,sy_tmag,sy_tmagerr1,sy_tmagerr2,sy_tmaglim,sy_tmagsymerr,sy_tmagstr,sy_tmagformat,sy_tmag_solnid,sy_tmag_reflink,sy_name,pl_controv_flag,pl_orbtper_systemref,pl_tranmid_systemref,st_metratio,st_spectype,st_spectype_solnid,st_spectype_reflink,sy_plxlim,sy_kepmag,sy_kepmagerr1,sy_kepmagerr2,sy_kepmaglim,sy_kepmagsymerr,sy_kepmagstr,sy_kepformat,sy_kepmag_solnid,sy_kepmag_reflink,st_rotp,st_rotperr1,st_rotperr2,st_rotpsymerr,st_rotplim,st_rotpstr,st_rotpformat,st_rotp_solnid,st_rotp_reflink,pl_projobliq,pl_projobliqerr1,pl_projobliqerr2,pl_projobliqsymerr,pl_projobliqlim,pl_projobliqstr,pl_projobliqformat,pl_denserr2,pl_denssymerr,pl_denslim,pl_densstr,pl_densformat,pl_dens_solnid,pl_dens_reflink,pl_trandep,pl_trandeperr1,pl_trandeperr2,pl_trandepsymerr,pl_trandeplim,pl_trandepstr,pl_trandepformat,pl_trandep_solnid,pl_trandep_reflink,pl_tranmid,pl_tranmiderr1,pl_tranmiderr2,pl_tranmidsymerr,pl_tranmidlim,pl_tranmidstr,pl_tranmidformat,pl_tranmid_solnid,pl_tranmid_reflink,pl_trandur,pl_trandurerr1,pl_trandurerr2,pl_tr

In [38]:
import pyvo

eu = pyvo.dal.TAPService("http://voparis-tap-planeto.obspm.fr/tap")
eu_t = eu.search("SELECT * FROM exoplanet.epn_core").to_table()

In [43]:
print(len(eu_t.colnames))
eu_t.colnames

136


['granule_uid',
 'granule_gid',
 'obs_id',
 'dataproduct_type',
 'target_name',
 'target_class',
 'time_min',
 'time_max',
 'time_sampling_step_min',
 'time_sampling_step_max',
 'time_exp_min',
 'time_exp_max',
 'spectral_range_min',
 'spectral_range_max',
 'spectral_sampling_step_min',
 'spectral_sampling_step_max',
 'spectral_resolution_min',
 'spectral_resolution_max',
 'c1min',
 'c1max',
 'c2min',
 'c2max',
 'c3min',
 'c3max',
 's_region',
 'c1_resol_min',
 'c1_resol_max',
 'c2_resol_min',
 'c2_resol_max',
 'c3_resol_min',
 'c3_resol_max',
 'spatial_frame_type',
 'incidence_min',
 'incidence_max',
 'emergence_min',
 'emergence_max',
 'phase_min',
 'phase_max',
 'instrument_host_name',
 'instrument_name',
 'measurement_type',
 'processing_level',
 'creation_date',
 'modification_date',
 'release_date',
 'service_title',
 'time_scale',
 'publisher',
 'bib_reference',
 'target_region',
 'species',
 'detection_type',
 'publication_status',
 'mass',
 'mass_error_min',
 'mass_error_max',